# Fix bugs in data merging of ESCO and O*NET
Felix Zaussinger | 13.04.2022

## Core Analysis Goal(s)
1. Fix bugs in occupation and skills metadata merging.
2. Gain confidence that the final datasets are valid.
3.

## Key Insight(s)
1.
2.
3.

In [1]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

from src import utils
# import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("poster")
sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

In [2]:
from src import utils, stats_utils, plotting_utils
from src.data.framework import Esco, Onet, Crosswalks

Code ...

In [3]:
esco = Esco()
crosswalks = Crosswalks()
onet = Onet()

### Checking problems with ONET-ESCO crosswalk
- some ONET greening occupations missing in Nesta crosswalk

In [4]:
gt_esco = esco.read_greenness_task_based()

In [5]:
gt_esco.title_gtp.unique()

array(['Chief Sustainability Officers', 'Green Marketers',
       'Geothermal Production Managers', 'Biofuels Production Managers',
       'Biomass Production Managers',
       'Methane/Landfill Gas Collection System Operators',
       'Hydroelectric Production Managers',
       'Biofuels/Biodiesel Technology and Product Development Managers',
       'Water Resource Specialists', 'Wind Energy Operations Managers',
       'Wind Energy Project Managers',
       'Brownfield Redevelopment Specialists and Site Managers',
       'Energy Auditors', 'Sustainability Specialists',
       'Environmental Engineers', 'Water/Wastewater Engineers',
       'Fuel Cell Engineers', 'Energy Engineers', 'Wind Energy Engineers',
       'Solar Energy Systems Engineers',
       'Environmental Engineering Technicians', 'Fuel Cell Technicians',
       'Soil and Water Conservationists', 'Climate Change Analysts',
       'Environmental Restoration Planners', 'Industrial Ecologists',
       'Environmental Economis

63 ONET greening occupations dont have a match to ESCO based on current crosswalk

In [6]:
bsel = gt_esco.loc[:, ["preferred_label"]].isna().values
df_no_match = gt_esco.loc[bsel]
df_no_match

,onet_code,title_gtp,occupation_type,n_new_green_tasks_gtp,n_existing_green_tasks_gtp,n_non_green_tasks_gtp,share_green_gtp,title_vona2018,share_green_vona2018,total_spec_tasks_vona2018,green_spec_tasks_vona2018,id,concept_uri,preferred_label,isco_level_4,onet_occupation
1,11-2011.01,Green Marketers,New Green N&E,16,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11-3051.02,Geothermal Production Managers,New Green N&E,17,0,0,1.000000,Geothermal Production Managers,1.000000,17.0,17.0,NaN,NaN,NaN,NaN,NaN
3,11-3051.03,Biofuels Production Managers,New Green N&E,14,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11-3051.04,Biomass Production Managers,New Green N&E,18,0,0,1.000000,Biomass Power Plant Managers,1.000000,18.0,18.0,NaN,NaN,NaN,NaN,NaN
5,11-3051.05,Methane/Landfill Gas Collection System Operators,New Green N&E,21,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,11-3051.06,Hydroelectric Production Managers,New Green N&E,19,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,11-9041.01,Biofuels/Biodiesel Technology and Product Deve...,New Green N&E,19,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,11-9121.02,Water Resource Specialists,New Green N&E,21,0,0,1.000000,Water Resource Specialists,1.000000,21.0,21.0,NaN,NaN,NaN,NaN,NaN
9,11-9199.09,Wind Energy Operations Managers,New Green N&E,16,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,11-9199.10,Wind Energy Project Managers,New Green N&E,15,0,0,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
gt_esco.columns

Index(['onet_code', 'title_gtp', 'occupation_type', 'n_new_green_tasks_gtp',
       'n_existing_green_tasks_gtp', 'n_non_green_tasks_gtp',
       'share_green_gtp', 'title_vona2018', 'share_green_vona2018',
       'total_spec_tasks_vona2018', 'green_spec_tasks_vona2018', 'id',
       'concept_uri', 'preferred_label', 'isco_level_4', 'onet_occupation'],
      dtype='object')

Summary: number of ESCO occupations matched to a single greening ONET occupation

In [8]:
grouping_cols = ["onet_code", "title_gtp", "occupation_type", "share_green_gtp"]

match_summary = gt_esco.groupby(grouping_cols)["preferred_label"].count().sort_values(ascending=False)
match_summary.rename("n_matches_esco", inplace=True)

match_summary.to_csv(
    os.path.join(useful_paths.results_dir, "validation", "onet_esco_crosswalk", "onet_to_esco_match_summary.csv"),
    sep=";"
)

match_summary

onet_code   title_gtp                                                                                            occupation_type        share_green_gtp
13-1022.00  Wholesale and Retail Buyers, Except Farm Products                                                    Green Enhanced Skills  0.300000           41
17-2141.00  Mechanical Engineers                                                                                 Green Enhanced Skills  0.285714           36
51-9012.00  Separating, Filtering, Clarifying, Precipitating, and Still Machine Setters, Operators, and Tenders  Green Enhanced Skills  0.047619           29
51-9061.00  Inspectors, Testers, Sorters, Samplers, and Weighers                                                 Green Enhanced Skills  0.062500           28
27-3022.00  Reporters and Correspondents                                                                         Green Enhanced Skills  0.035714           18
                                                          

In [9]:
missing_matches = match_summary[match_summary == 0].reset_index()
missing_matches

,onet_code,title_gtp,occupation_type,share_green_gtp,n_matches_esco
0,11-9199.11,Brownfield Redevelopment Specialists and Site ...,New Green N&E,1.000000,0
1,11-9199.10,Wind Energy Project Managers,New Green N&E,1.000000,0
2,11-9199.09,Wind Energy Operations Managers,New Green N&E,1.000000,0
3,49-9042.00,"Maintenance and Repair Workers, General",Green Enhanced Skills,0.137931,0
4,47-4099.01,Solar Photovoltaic Installers,New Green N&E,1.000000,0
5,47-4099.02,Solar Thermal Installers and Technicians,New Green N&E,1.000000,0
6,13-1073.00,Training and Development Specialists,Green Enhanced Skills,0.100000,0
7,13-1081.01,Logistics Engineers,Existing Green N&E,0.366667,0
8,47-5041.00,Continuous Mining Machine Operators,Green Enhanced Skills,0.133333,0
9,11-9121.02,Water Resource Specialists,New Green N&E,1.000000,0


63 occupations are completely missing any match to ESCO occupations. What is going on there?

In [10]:
missing_matches.shape

(63, 5)

Inspecting the crosswalk I used for the thesis

In [11]:
cw = crosswalks.onet_esco_mcc_full
cw

,id,concept_uri,preferred_label,isco_level_4,onet_code,onet_occupation,onet_code_6d
0,0,http://data.europa.eu/esco/occupation/00030d09...,technical director,2166,27-1011.00,art directors,27-1011
1,1,http://data.europa.eu/esco/occupation/000e93a3...,metal drawing machine operator,8121,51-4021.00,"extruding and drawing machine setters, operato...",51-4021
2,2,http://data.europa.eu/esco/occupation/0019b951...,precision device inspector,7543,51-9061.00,"inspectors, testers, sorters, samplers, and we...",51-9061
3,3,http://data.europa.eu/esco/occupation/0022f466...,air traffic safety technician,3155,17-3023.01,electronics engineering technicians,17-3023
4,4,http://data.europa.eu/esco/occupation/002da35b...,hospitality revenue manager,2431,13-1161.00,market research analysts and marketing special...,13-1161
...,...,...,...,...,...,...,...
2937,2937,http://data.europa.eu/esco/occupation/ff656b3a...,demographer,2120,15-2041.00,statisticians,15-2041
2938,2938,http://data.europa.eu/esco/occupation/ff8d4065...,sorter labourer,9612,51-9199.01,recycling and reclamation workers,51-9199
2939,2939,http://data.europa.eu/esco/occupation/ffa4dd5d...,armoured car guard,5414,33-9032.00,security guards,33-9032
2940,2940,http://data.europa.eu/esco/occupation/ffade2f4...,civil service administrative officer,2422,11-3011.00,administrative services managers,11-3011


2942 ESCO occupations are matched to 669 ONET occupations

In [12]:
cw.onet_code.unique().shape

(669,)

None of these 63 onet occupations is part of this crosswalk

In [13]:
for code, label in zip(missing_matches.onet_code, missing_matches.title_gtp):
    match_original_code = code in cw.onet_code.values
    print("{} ¦ {} ¦ {}".format(code, label, match_original_code))

11-9199.11 ¦ Brownfield Redevelopment Specialists and Site Managers ¦ False
11-9199.10 ¦ Wind Energy Project Managers ¦ False
11-9199.09 ¦ Wind Energy Operations Managers ¦ False
49-9042.00 ¦ Maintenance and Repair Workers, General ¦ False
47-4099.01 ¦ Solar Photovoltaic Installers ¦ False
47-4099.02 ¦ Solar Thermal Installers and Technicians ¦ False
13-1073.00 ¦ Training and Development Specialists ¦ False
13-1081.01 ¦ Logistics Engineers ¦ False
47-5041.00 ¦ Continuous Mining Machine Operators ¦ False
11-9121.02 ¦ Water Resource Specialists ¦ False
11-9041.01 ¦ Biofuels/Biodiesel Technology and Product Development Managers ¦ False
11-9199.06 ¦ Logistics Managers ¦ False
17-3029.03 ¦ Electromechanical Engineering Technologists ¦ False
49-9099.01 ¦ Geothermal Technicians ¦ False
49-9099.02 ¦ Wind Turbine Service Technicians ¦ False
11-9041.00 ¦ Engineering Managers ¦ False
11-9013.02 ¦ Farm and Ranch Managers ¦ False
51-8011.00 ¦ Nuclear Power Reactor Operators ¦ False
51-8099.01 ¦ Bio

#### Option 1: assign value to coarser SOC level and check if there is a match

In [14]:
cw.query("onet_code == '51-4021.00'")
cw_unique_onet = cw.drop_duplicates(subset="onet_code")

sel = cw_unique_onet.onet_code == "11-9199.00"
cw_unique_onet.loc[sel, 'onet_occupation'].values.tolist()[0]

'managers, all other'

In [15]:
summary_dict = {
    "onet_code_orig": [],
    "onet_label_orig": [],
    "onet_match_orig": [],
    "onet_code_1_below": [],
    "onet_label_1_below": [],
    "onet_match_1_below": [],
    "onet_code_2_below": [],
    "onet_label_2_below": [],
    "onet_match_2_below": []
}

for code_orig, label_orig in zip(missing_matches.onet_code, missing_matches.title_gtp):
    # coarser onet soc codes
    onet_code_1_below = code_orig[:-1] + "0"
    onet_code_2_below = code_orig[:-2] + "00"

    # original match with cw?
    match_orig = code_orig in cw_unique_onet.onet_code.values

    # if not, check one level below if not
    if match_orig is False:

        onet_match_1_below = onet_code_1_below in cw_unique_onet.onet_code.values
        if onet_match_1_below:
            sel = cw_unique_onet.onet_code == onet_code_1_below
            onet_label_1_below = cw_unique_onet.loc[sel, 'onet_occupation'].values.tolist()[0]

            onet_code_2_below = False
            onet_label_2_below = False
            onet_match_2_below = False
        else:
            onet_label_1_below = None

            # check if there is a match two levels below
            onet_match_2_below = onet_code_2_below in cw_unique_onet.onet_code.values
            if onet_match_2_below:
                sel_2 = cw_unique_onet.onet_code == onet_code_2_below
                onet_label_2_below = cw_unique_onet.loc[sel_2, 'onet_occupation'].values.tolist()[0]
                pass

    else:
        onet_match_1_below = None
        onet_label_1_below = None

    # update dict
    summary_dict["onet_code_orig"].append(code_orig)
    summary_dict["onet_label_orig"].append(label_orig)
    summary_dict["onet_match_orig"].append(match_orig)

    summary_dict["onet_code_1_below"].append(onet_code_1_below)
    summary_dict["onet_label_1_below"].append(onet_label_1_below)
    summary_dict["onet_match_1_below"].append(onet_match_1_below)

    summary_dict["onet_code_2_below"].append(onet_code_2_below)
    summary_dict["onet_label_2_below"].append(onet_label_2_below)
    summary_dict["onet_match_2_below"].append(onet_match_2_below)


df_cw_coarsening_summary = pd.DataFrame.from_dict(summary_dict)

df_cw_coarsening_summary

,onet_code_orig,onet_label_orig,onet_match_orig,onet_code_1_below,onet_label_1_below,onet_match_1_below,onet_code_2_below,onet_label_2_below,onet_match_2_below
0,11-9199.11,Brownfield Redevelopment Specialists and Site ...,False,11-9199.10,None,False,11-9199.00,"managers, all other",True
1,11-9199.10,Wind Energy Project Managers,False,11-9199.10,None,False,11-9199.00,"managers, all other",True
2,11-9199.09,Wind Energy Operations Managers,False,11-9199.00,"managers, all other",True,False,False,False
3,49-9042.00,"Maintenance and Repair Workers, General",False,49-9042.00,None,False,49-9042.00,False,False
4,47-4099.01,Solar Photovoltaic Installers,False,47-4099.00,None,False,47-4099.00,False,False
5,47-4099.02,Solar Thermal Installers and Technicians,False,47-4099.00,None,False,47-4099.00,False,False
6,13-1073.00,Training and Development Specialists,False,13-1073.00,None,False,13-1073.00,False,False
7,13-1081.01,Logistics Engineers,False,13-1081.00,None,False,13-1081.00,False,False
8,47-5041.00,Continuous Mining Machine Operators,False,47-5041.00,None,False,47-5041.00,False,False
9,11-9121.02,Water Resource Specialists,False,11-9121.00,natural sciences managers,True,False,False,False


Results of downwalking
- 24 additional matches through downwalking to SOC 6D level
- 22 additional matches one level below (SOC 7D), 2 matches two levels below (SOC 8D)

In [16]:
df_cw_coarsening_summary.onet_match_1_below.sum()

22

In [17]:
df_cw_coarsening_summary.onet_match_2_below.sum()

2

#### Option 2: use reduced crosswalk at ESCO LVL 5
Idea
- use reduced crosswalk file used in MCC report that matches at the ESCO lvl 5.
- Assign same membership to all lower-level children.

Conclusion
- no additional matches are found, the crosswalk would need to be reconstructed.

In [18]:
esco5_onet_cw = crosswalks.onet_esco_mcc_reduced
esco5_onet_cw.concept_uri.unique()

array(['http://data.europa.eu/esco/occupation/0ee67d5e-8de9-48f0-97c5-ba23d486b1e9',
       'http://data.europa.eu/esco/occupation/404a50e9-61ce-448d-a695-bbfc697a0727',
       'http://data.europa.eu/esco/occupation/5bca96fd-4203-400b-be58-9e9b50bc98da',
       ...,
       'http://data.europa.eu/esco/occupation/c28cb56b-003e-499a-8dbf-b654864e7867',
       'http://data.europa.eu/esco/occupation/e16e4cb0-6f7d-4bf9-8f90-d1d866ad229b',
       'http://data.europa.eu/esco/occupation/e31c83ac-cb74-4936-88f1-90ed0997b29b'],
      dtype=object)

In [19]:
matches =  []
for code_orig, label_orig in zip(missing_matches.onet_code, missing_matches.title_gtp):
    match = esco5_onet_cw.loc[esco5_onet_cw.loc[:, "onet_code"] == code_orig]
    matches.append(not match.empty)
    print(code_orig, "¦", label_orig, "¦", not match.empty)

11-9199.11 ¦ Brownfield Redevelopment Specialists and Site Managers ¦ False
11-9199.10 ¦ Wind Energy Project Managers ¦ False
11-9199.09 ¦ Wind Energy Operations Managers ¦ False
49-9042.00 ¦ Maintenance and Repair Workers, General ¦ False
47-4099.01 ¦ Solar Photovoltaic Installers ¦ False
47-4099.02 ¦ Solar Thermal Installers and Technicians ¦ False
13-1073.00 ¦ Training and Development Specialists ¦ False
13-1081.01 ¦ Logistics Engineers ¦ False
47-5041.00 ¦ Continuous Mining Machine Operators ¦ False
11-9121.02 ¦ Water Resource Specialists ¦ False
11-9041.01 ¦ Biofuels/Biodiesel Technology and Product Development Managers ¦ False
11-9199.06 ¦ Logistics Managers ¦ False
17-3029.03 ¦ Electromechanical Engineering Technologists ¦ False
49-9099.01 ¦ Geothermal Technicians ¦ False
49-9099.02 ¦ Wind Turbine Service Technicians ¦ False
11-9041.00 ¦ Engineering Managers ¦ False
11-9013.02 ¦ Farm and Ranch Managers ¦ False
51-8011.00 ¦ Nuclear Power Reactor Operators ¦ False
51-8099.01 ¦ Bio

In [20]:
np.array(matches).sum()

0

#### Option 3: use ONET-SOC and SOC-ISCO crosswalks from MCC project (for LFS data only)

Idea:
- don't go via ESCO but directly match the GTP data to ISCO occupations

Conclusions:
- 56/63 occupations can be matched in this way.
- Not attaching to ESCO, however, will make us loose a lot of information when working with the national (DE/IT) data.

In [21]:
crosswalks.soc10_isco08_ibs

,soc10,isco08
0,111011,1112
1,111011,1113
2,111011,1120
3,111021,1112
4,111021,1114
...,...,...
1126,553015,310
1127,553016,310
1128,553017,310
1129,553018,310


In [22]:
onet_us2010soc = crosswalks.onet_soc10
onet_us2010soc

,O*NET-SOC 2010 Code,O*NET-SOC 2010 Title,2010 SOC Code,2010 SOC Title
0,11-1011.00,Chief Executives,11-1011,Chief Executives
1,11-1011.03,Chief Sustainability Officers,11-1011,Chief Executives
2,11-1021.00,General and Operations Managers,11-1021,General and Operations Managers
3,11-1031.00,Legislators,11-1031,Legislators
4,11-2011.00,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers
...,...,...,...,...
1105,55-3015.00,Command and Control Center Specialists,55-3015,Command and Control Center Specialists
1106,55-3016.00,Infantry,55-3016,Infantry
1107,55-3017.00,Radar and Sonar Technicians,55-3017,Radar and Sonar Technicians
1108,55-3018.00,Special Forces,55-3018,Special Forces


In [23]:
onet_us2010soc.iloc[:, 2].unique()

array(['11-1011', '11-1021', '11-1031', '11-2011', '11-2021', '11-2022',
       '11-2031', '11-3011', '11-3021', '11-3031', '11-3051', '11-3061',
       '11-3071', '11-3111', '11-3121', '11-3131', '11-9013', '11-9021',
       '11-9031', '11-9032', '11-9033', '11-9039', '11-9041', '11-9051',
       '11-9061', '11-9071', '11-9081', '11-9111', '11-9121', '11-9131',
       '11-9141', '11-9151', '11-9161', '11-9199', '13-1011', '13-1021',
       '13-1022', '13-1023', '13-1031', '13-1032', '13-1041', '13-1051',
       '13-1071', '13-1074', '13-1075', '13-1081', '13-1111', '13-1121',
       '13-1131', '13-1141', '13-1151', '13-1161', '13-1199', '13-2011',
       '13-2021', '13-2031', '13-2041', '13-2051', '13-2052', '13-2053',
       '13-2061', '13-2071', '13-2072', '13-2081', '13-2082', '13-2099',
       '15-1111', '15-1121', '15-1122', '15-1131', '15-1132', '15-1133',
       '15-1134', '15-1141', '15-1142', '15-1143', '15-1151', '15-1152',
       '15-1199', '15-2011', '15-2021', '15-2031', 

How many matches do I find in the ONET-SOC crosswalk?

In [24]:
matches =  []
for code_orig, label_orig in zip(missing_matches.onet_code, missing_matches.title_gtp):
    match = onet_us2010soc.loc[onet_us2010soc.loc[:, 'O*NET-SOC 2010 Code'] == code_orig]
    matches.append(not match.empty)
    print(code_orig, "¦", label_orig, "¦", not match.empty)

11-9199.11 ¦ Brownfield Redevelopment Specialists and Site Managers ¦ True
11-9199.10 ¦ Wind Energy Project Managers ¦ True
11-9199.09 ¦ Wind Energy Operations Managers ¦ True
49-9042.00 ¦ Maintenance and Repair Workers, General ¦ False
47-4099.01 ¦ Solar Photovoltaic Installers ¦ False
47-4099.02 ¦ Solar Thermal Installers and Technicians ¦ True
13-1073.00 ¦ Training and Development Specialists ¦ False
13-1081.01 ¦ Logistics Engineers ¦ True
47-5041.00 ¦ Continuous Mining Machine Operators ¦ True
11-9121.02 ¦ Water Resource Specialists ¦ True
11-9041.01 ¦ Biofuels/Biodiesel Technology and Product Development Managers ¦ True
11-9199.06 ¦ Logistics Managers ¦ False
17-3029.03 ¦ Electromechanical Engineering Technologists ¦ True
49-9099.01 ¦ Geothermal Technicians ¦ True
49-9099.02 ¦ Wind Turbine Service Technicians ¦ False
11-9041.00 ¦ Engineering Managers ¦ True
11-9013.02 ¦ Farm and Ranch Managers ¦ True
51-8011.00 ¦ Nuclear Power Reactor Operators ¦ True
51-8099.01 ¦ Biofuels Process

56 of the 63 matchless occupation can be matched!

In [25]:
np.array(matches).sum()

56

More detailed analysis

#### Option 4: use ONET SOC to ISCO crosswalk from the Institute for Structural Research - IBS

Idea:
- use an existing crosswalk from ONET-SOC to ISCO
-

In [26]:
soc10_isco08 = crosswalks.soc10_isco08_ibs
soc10_isco08

,soc10,isco08
0,111011,1112
1,111011,1113
2,111011,1120
3,111021,1112
4,111021,1114
...,...,...
1126,553015,310
1127,553016,310
1128,553017,310
1129,553018,310


In [27]:
soc10_isco08.soc10.unique()

array([111011, 111021, 111031, 112011, 112021, 112022, 112031, 113011,
       113021, 113031, 113051, 113061, 113071, 113111, 113121, 113131,
       119013, 119021, 119031, 119032, 119033, 119039, 119041, 119051,
       119061, 119071, 119081, 119111, 119121, 119131, 119141, 119151,
       119161, 119199, 131011, 131021, 131022, 131023, 131031, 131032,
       131041, 131051, 131071, 131074, 131075, 131081, 131111, 131121,
       131131, 131141, 131151, 131161, 131199, 132011, 132021, 132031,
       132041, 132051, 132052, 132053, 132061, 132071, 132072, 132081,
       132082, 132099, 151111, 151121, 151122, 151131, 151132, 151133,
       151134, 151141, 151142, 151143, 151151, 151152, 151199, 152011,
       152021, 152031, 152041, 152091, 152099, 171011, 171012, 171021,
       171022, 172011, 172021, 172031, 172041, 172051, 172061, 172071,
       172072, 172081, 172111, 172112, 172121, 172131, 172141, 172151,
       172161, 172171, 172199, 173011, 173012, 173013, 173019, 173021,
      

## Green/brown occupation master file
- triangulation via tasks and skills

In [28]:
smd = esco.combine_skills_metadata(variable_selection=None)
omd = esco.combine_occupation_metadata(skills_metadata=smd)

C:\Users\fzaussinger\Miniconda3\envs\re4gt\lib\site-packages\openpyxl\worksheet\_reader.py:312: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
C:\Users\fzaussinger\Miniconda3\envs\re4gt\lib\site-packages\openpyxl\worksheet\_reader.py:312: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
T:\Documents\Projects\04_jrc_green-skills-regional\03_data-analysis\re4gt\src\data\framework.py:1435: RuntimeWarning: invalid value encountered in true_divide
  occ_share = n_gbn_specific_skills / n_total_specific_skills


In [31]:
omd

,conceptType,conceptUri,iscoGroup,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,regulatedProfessionNote,scopeNote,definition,inScheme,description,code,isco_level_4,isco_level_1,isco_level_2,isco_level_3,isco_label_1,isco_label_2,isco_label_3,isco_label_4,share_green_esco,share_brown_esco,share_neutral_esco,gbn_classification_esco,share_green_esco_ess,share_brown_esco_ess,share_neutral_esco_ess,gbn_classification_esco_ess
0,Occupation,http://data.europa.eu/esco/occupation/00030d09...,2654,technical director,technical and operations director\nhead of tec...,NaN,released,2016-07-05T13:58:41Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Technical directors realise the artistic visio...,2654.1.7,2654,2,26,265,Professionals,"Legal, social and cultural professionals",Creative and performing artists,"Film, stage and related directors and producers",0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral
1,Occupation,http://data.europa.eu/esco/occupation/1a7fb683...,2654,video and motion picture director,television director\nfilm maker\nseries direct...,NaN,released,2017-01-17T13:59:29Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Video and motion picture directors are respons...,2654.1.8,2654,2,26,265,Professionals,"Legal, social and cultural professionals",Creative and performing artists,"Film, stage and related directors and producers",0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral
2,Occupation,http://data.europa.eu/esco/occupation/2f372afe...,2654,performance lighting director,television lighting supervisor\nTV lighting su...,NaN,released,2016-07-05T13:46:52Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Performance lighting directors determine what ...,2654.1.4,2654,2,26,265,Professionals,"Legal, social and cultural professionals",Creative and performing artists,"Film, stage and related directors and producers",0.052632,0.0,0.947368,neutral,0.055556,0.0,0.944444,neutral
3,Occupation,http://data.europa.eu/esco/occupation/30b25ee4...,2654,animation director,animation process director\ncartoon animation ...,NaN,released,2021-03-31T15:31:28.501Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Animation directors supervise and recruit mult...,2654.1.1,2654,2,26,265,Professionals,"Legal, social and cultural professionals",Creative and performing artists,"Film, stage and related directors and producers",0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral
4,Occupation,http://data.europa.eu/esco/occupation/3b6bea7d...,2654,video and motion picture producer,video & motion picture producer\nmovie produce...,NaN,released,2017-01-17T13:58:37Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/memb...,Video and motion picture producers supervise t...,2654.3.2,2654,2,26,265,Professionals,"Legal, social and cultural professionals",Creative and performing artists,"Film, stage and related directors and producers",0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3003,Occupation,http://data.europa.eu/esco/occupation/b059d331...,9520,hawker,street trader\nmarket stall trader\ncar-boot s...,NaN,released,2021-10-06T08:30:39.429Z,http://data.europa.eu/esco/regulated-professio...,NaN,NaN,http://data.europa.eu/esco/concept-scheme/occu...,Hawkers sell goods and services on established...,9520.1,9520,9,95,952,Elementary occupations,Street and related sales and service workers,Street vendors (excluding food),Street vendors (excluding food),0.000000,0.0,1.000000,neutral,0.000000,0.0,1.000000,neutral
3004,Occupation,http://data.europa.eu/esco/occupation/96a0d8d3...,9411,quick service restaurant crew m

1 ONET: Vona 2018 (Greenness), SOC 8-digit matched via ONET-ESCO crosswalk [CHECK]
2 ONET: GTP 2011 (Greenness), SOC 8-digit matched via ONET-ESCO crosswalk [CHECK]
3 ONET: Vona 2019 (Greenness), SOC 6-digit matched via IBS SOC-ISCO crosswalk
4 ONET: JRC/Consoli (Greenness), CP-2011 5-digit matched via CP2011-ESCO crosswalk
5 ESCO: ESCO 2022 (Greenness) [CHECK]

In [29]:
cols = ['conceptUri', 'preferredLabel',
       'isco_level_4', 'isco_level_1', 'isco_level_2', 'isco_level_3',
       'isco_label_1', 'isco_label_2', 'isco_label_3', 'isco_label_4',
       'share_green_esco', 'share_brown_esco', 'gbn_classification_esco',
       'share_green_esco_ess', 'share_brown_esco_ess', 'gbn_classification_esco_ess',
       'onet_code', 'title_gtp', 'share_green_gtp', 'share_green_vona2018',
       'is_brown_onet', 'is_green_onet', 'gbn_classification_onet']

In [30]:
omd_sub = omd[cols]

KeyError: "['onet_code', 'title_gtp', 'share_green_gtp', 'share_green_vona2018', 'is_brown_onet', 'is_green_onet', 'gbn_classification_onet'] not in index"

Fill nan values in ONET greenness shares with zeros

In [ ]:
omd_sub.loc[:, "share_green_gtp"] = omd_sub.share_green_gtp.fillna(0)
omd_sub.loc[:, "share_green_vona2018"] = omd_sub.share_green_vona2018.fillna(0)

**Triangulation of green occupations based on intersecting task- and skill data**

Distribution of shares, descriptive statistics

In [ ]:
omd_sub.describe()

In [ ]:
omd_sub.hist(bins=20, log=False, sharex=True)
plt.tight_layout()

Distributions quite skewed, use median

In [ ]:
omd_sub.select_dtypes("float").apply(stats_utils.naniqr, axis=0)

In [ ]:
omd_sub.select_dtypes("float").quantile(0.90)

In [ ]:
omd_sub.loc[omd_sub.loc[:, "share_green_esco"] > 0.2]

In [ ]:
omd_sub.select_dtypes("float")

In [ ]:
omd_sub[(omd_sub.share_green_esco_ess > 0) & ((omd_sub.share_green_gtp > 0) | (omd_sub.share_green_vona2018 > 0))]

In [ ]:
omd_sub[(omd_sub.share_brown_esco > 0) & omd_sub.is_brown_onet]